In [35]:
import os
import sys
from tqdm import tqdm
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.optim import SGD, Adam
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

In [36]:
torch.cuda.is_available()

True

In [37]:
base_url = os.getcwd()
sys.path.append(base_url + '/Util')

In [38]:
from Visualization import *
from DataPreprocess import *
from Miscellaneous import *

# Intended Task

In [39]:
TASKS = ["Cross-Position", "Cross-Person", "Cross-Device"]
task = None
while True:
    print("1. Cross-Position\n2. Cross-Person\n3. Cross-Device")
    user_input = input("Please enter your task: ")
    if user_input == "1":
        task = "Cross-Position"
    elif user_input == "2":
        task = "Cross-Person"
    elif user_input == "3":
        task = "Cross-Device"
    if task in TASKS:
        break
    else:
        print("Invalid task. Please choose a valid task.")
print(f"Chosen task: {task}")

1. Cross-Position
2. Cross-Person
3. Cross-Device
Please enter your task: 1
Chosen task: Cross-Position


# Dataset Selection

In [40]:
DATASETS = ["OPP", "DSADS", "PAMAP", "WISDM", "HHAR"]
dataset = None
if task == 'Cross-Position':
    while True:
        print("1. OPP\n2. DSADS\n3. PAMAP\n")
        user_input = input("Please enter your dataset: ")
        if user_input == "1":
            dataset = "OPP"
        elif user_input == "2":
            dataset = "DSADS"
        elif user_input == "3":
            dataset = "PAMAP"
        if dataset in DATASETS:
            break
        else:
            print("Invalid dataset. Please choose a valid dataset.")
elif task == 'Cross-Person':
    while True:
        print("1. OPP\n2. DSADS\n3. PAMAP\n4. WISDM\n")
        user_input = input("Please enter your dataset: ")
        if user_input == "1":
            dataset = "OPP"
        elif user_input == "2":
            dataset = "DSADS"
        elif user_input == "3":
            dataset = "PAMAP"
        elif user_input == "4":
            dataset = "WISDM"
        if dataset in DATASETS:
            break
        else:
            print("Invalid dataset. Please choose a valid dataset.")
print(f"Chosen dataset: {dataset}")

1. OPP
2. DSADS
3. PAMAP

Please enter your dataset: 2
Chosen dataset: DSADS


# Reproduciblitiy (Must Change Before Each Experiment)

In [41]:
os.environ["CUBLAS_WORKSPACE_CONFIG"]=":4096:8"

def set_deterministic_and_get_rng(seed):
    """
    Set numpy, pytorch, and cudnn to be fully reproducible across runs
    and get rng  for predictable dataloaders

    https://pytorch.org/docs/stable/notes/randomness.html

    :return:
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True)

    rng = torch.Generator()
    rng.manual_seed(seed)

    def seed_worker(worker_id):
        worker_seed = torch.initial_seed() % 2 ** 32
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    return rng, seed_worker

seeds = [0, 1, 42, 123, 1234]
seed = seeds[0] # Change this
rng, seed_worker = set_deterministic_and_get_rng(seed=seed)

In [42]:
# All of these have to taken input from shell script
win_size = 128
overlap = 0.875

if task == "Cross-Position":
    if dataset == "DSADS":
        domains = {0:"TORSO", 1:"RA", 2:"LA", 3:"RL", 4:"LL"}
        sources = [1, 2, 3, 4]
        nsources = len(sources)
        target = 0
        activities = [ "standing", "lying-back", "ascending", "walking-parking-lot", "treadmill-running", "stepper-exercise", "cross-trainer-exercise", "rowing", "jumping",  "playing-basketball"]
        activity_num = len(activities)
        item = ["train","valid","test"]
        person_list = ["User1","User2","User3","User4", "User5","User6","User7","User8"]
        position_array = ["TORSO","RA","LA","RL","LL"]
    elif dataset == "OPP":
        domains = {0:"BACK", 1:"RUA", 2:"RLA", 3:"LUA", 4:"LLA"}
        sources = [1, 2, 3, 4]
        nsources = len(sources)
        target = 0
        activities = ['Sitting','Standing','Walking','Running']
        activity_num = len(activities)
        item = ["train","valid","test"]
        person_list = ["U1","U2","U3","U4"]
        position_array = ["BACK", "RUA", "RLA", "LUA","LLA"]
    elif dataset == "PAMAP":
        domains = {0:"Wrist", 1:"Chest", 2:"Ankle"}
        sources = [0, 2]
        nsources = len(sources)
        target = 1
        activities = ['ascending', 'lying', 'sitting', 'standing', 'walking', 'descending', 'vacuum', 'ironing', 'running', 'cycling', 'rope_jumping']
        activity_num = len(activities)
        item = ["train","valid","test"]
        person_list = ["U1","U2","U3","U4","U5","U6","U7","U8"]
        position_array = ['Wrist', 'Chest', 'Ankle']
        
elif task == "Cross-Person":
    if dataset == "DSADS":
        domains = {0:"User1", 1:"User2", 2:"User3", 3:"User4", 4:"User5", 5:"User6", 6:"User7", 7:"User8"}
        sources = [4, 5, 6, 7]
        target = [0, 1, 2, 3]
        nsources = len(sources)
        position_list = ["TORSO", "RA", "LA", "RL", "LL"]
        activities = [ "standing", "lying-back", "ascending", "walking-parking-lot", "treadmill-running", "stepper-exercise", "cross-trainer-exercise", "rowing", "jumping",  "playing-basketball"]
        activity_num = len(activities)
        item = ["train","valid","test"]
        person_array = ["User1","User2","User3","User4","User5","User6","User7","User8"]
    elif dataset == "OPP":
        domains = {0:"U1", 1:"U2", 2:"U3", 3:"U4"}
        sources = [0, 1]
        target = [2, 3]
        nsources = len(sources)
        position_list = ["BACK", "RUA", "RLA", "LUA","LLA"]
        activities = ['Sitting','Standing','Walking','Running']
        activity_num = len(activities)
        item = ["train","valid","test"]
        person_array = ["U1","U2","U3","U4"]
    elif dataset == "PAMAP":
        domains = {0:'U1', 1:'U2', 2:'U3', 3:'U4', 4:'U5', 5:'U6', 6:'U7', 7:'U8'}
        sources = [0, 1, 2, 3]
        nsources = len(sources)
        target = [4, 5, 6, 7]
        activities = ['ascending', 'lying', 'sitting', 'standing', 'walking', 'descending', 'vacuum', 'ironing', 'running', 'cycling', 'rope_jumping']
        activity_num = len(activities)
        item = ["train","valid","test"]
        position_list = ['Wrist', 'Chest', 'Ankle']
        person_array = ["U1","U2","U3","U4","U5","U6","U7","U8"]
elif task == "Cross-Device":
    pass

print(activity_num)

10


In [43]:
step_size=int(win_size*(1-overlap))
gpu_id= 0
DEVICE = torch.device('cuda:'+str(gpu_id) if torch.cuda.is_available() else 'cpu')

In [44]:
if dataset == "DSADS":
    AXIS = 3
    FROM = 0
    TO = FROM+3
    START = 4
    END = 5
elif dataset == "OPP":
    AXIS = 3
    FROM = 0
    TO = FROM+3
    START = 3
    END = 4
elif dataset == "PAMAP":
    AXIS = 3
    FROM = 0
    TO = FROM+3
    START = 4
    END = 5

In [45]:
if task == "Cross-Position":
    if dataset == "DSADS":
        folder_name = str(activity_num)+ "_Activity"+"_Window_"+str(win_size)+ "_Overlap_"+str(overlap)
        dataset_path = base_url + "/Dataset_Preprocessed/DSADS/Data Files/"+folder_name+"/"
        save_path = base_url + "/Output/DSADS-Cross-Position-Heterogeneity/"+folder_name+"/Target-Position-" + position_array[target] + '/seed' + str(seed) + "/"
        main_folder = "DSADS-AllPerson-DifferentPosition"
    elif dataset == "OPP":
        folder_name = str(activity_num)+ "_Activity"+"_Window_"+str(win_size)+ "_Overlap_"+str(overlap)
        dataset_path = base_url + "/Dataset_Preprocessed/OPPORTUNITY/Data Files/"+folder_name+"/"
        save_path = base_url + "/Output/OPP-Cross-Position-Heterogeneity/"+folder_name+"/Target-Position-" + position_array[target] + '/seed' + str(seed) + "/"
        main_folder = "OPP-AllPerson-DifferentPosition"
    elif dataset == "PAMAP":
        folder_name = str(activity_num)+ "_Activity"+"_Window_"+str(win_size)+ "_Overlap_"+str(overlap)
        dataset_path = base_url + "/Dataset_Preprocessed/PAMAP/Data Files/"+folder_name+"/"
        save_path = base_url + "/Output/PAMAP-Cross-Position-Heterogeneity/"+folder_name+"/Target-Position-" + position_array[target] + '/seed' + str(seed) + "/"
        main_folder = "PAMAP-AllPerson-DifferentPosition"
        
elif task == "Cross-Person":
    target_string = "_".join([domains[i] for i in target])
    if dataset == "DSADS":
        folder_name = str(activity_num)+ "_Activity"+"_Window_"+str(win_size)+ "_Overlap_"+str(overlap)
        dataset_path = base_url + "/Dataset_Preprocessed/DSADS/Data Files/"+folder_name+"/"
        save_path = base_url + "/Output/DSADS-Cross-Person-Heterogeneity/"+folder_name+"/Target-" + target_string + '/seed' + str(seed) + "/"
        main_folder = "DSADS-SamePosition-DifferentPerson"
    elif dataset == "OPP":
        folder_name = str(activity_num)+ "_Activity"+"_Window_"+str(win_size)+ "_Overlap_"+str(overlap)
        dataset_path = base_url + "/Dataset_Preprocessed/OPPORTUNITY/Data Files/"+folder_name+"/"
        save_path = base_url + "/Output/OPP-Cross-Person-Heterogeneity/"+folder_name+"/Target-" + target_string + '/seed' + str(seed) + "/"
        main_folder = "OPP-SamePosition-DifferentPerson"
    elif dataset == "PAMAP":
        folder_name = str(activity_num)+ "_Activity"+"_Window_"+str(win_size)+ "_Overlap_"+str(overlap)
        dataset_path = base_url + "/Dataset_Preprocessed/PAMAP/Data Files/"+folder_name+"/"
        save_path = base_url + "/Output/PAMAP-Cross-Person-Heterogeneity/"+folder_name+"/Target-" + target_string + '/seed' + str(seed) + "/"
        main_folder = "PAMAP-AllPerson-DifferentPerson"

if not os.path.exists(save_path):
    os.makedirs(save_path)

In [46]:
s_train = []
t_train = []

s_gt_train = []
t_gt_train = []

#===================================================#

s_valid = []
t_valid = []

s_gt_valid = []
t_gt_valid = []

#===================================================#

s_test = []
t_test = []

s_gt_test = []
t_gt_test = []

In [47]:
# sources can be [0, 1, 3, 4], and target = 2
if task == 'Cross-Position':
    print('Source')
    df_aggregated_train = pd.DataFrame()
    df_aggregated_valid = pd.DataFrame()
    df_aggregated_test = pd.DataFrame()
    for i in range(len(sources)):
        # Create an empty DataFrame to hold the aggregated data for this source position
        # Combine data from all users for the current source position
        for person in person_list:
            for split_index in range(0, 3):
                file_name = person + "_" + position_array[sources[i]] + '_' + item[split_index]
                print(f"Processing: {file_name}")
                df = pd.read_csv(dataset_path + file_name + '.csv', sep=",")
                
                # Aggregate data into the corresponding split DataFrame
                if split_index == 0:
                    df_aggregated_train = pd.concat([df_aggregated_train, df], ignore_index=True)
                elif split_index == 1:
                    df_aggregated_valid = pd.concat([df_aggregated_valid, df], ignore_index=True)
                elif split_index == 2:
                    df_aggregated_test = pd.concat([df_aggregated_test, df], ignore_index=True)
        
        # Pass the aggregated data to calculate_window
    calculate_window(df_aggregated_train, s_train, s_gt_train, win_size, step_size, FROM, TO, START, END, AXIS)
    calculate_window(df_aggregated_valid, s_valid, s_gt_valid, win_size, step_size, FROM, TO, START, END, AXIS)
    calculate_window(df_aggregated_test, s_test, s_gt_test, win_size, step_size, FROM, TO, START, END, AXIS)

    # For the target position, similarly aggregate data across all users
    print('Target')
    df_aggregated_train_target = pd.DataFrame()
    df_aggregated_valid_target = pd.DataFrame()
    df_aggregated_test_target = pd.DataFrame()
    for person in person_list:
        for split_index in range(0, 3):
            file_name = person + "_" + position_array[target] + '_' + item[split_index]
            print(f"Processing: {file_name}")
            df = pd.read_csv(dataset_path + file_name + '.csv', sep=",")
            
            if split_index == 0:
                df_aggregated_train_target = pd.concat([df_aggregated_train_target, df], ignore_index=True)
            elif split_index == 1:
                df_aggregated_valid_target = pd.concat([df_aggregated_valid_target, df], ignore_index=True)
            elif split_index == 2:
                df_aggregated_test_target = pd.concat([df_aggregated_test_target, df], ignore_index=True)

    calculate_window(df_aggregated_train_target, t_train, t_gt_train, win_size, step_size, FROM, TO, START, END, AXIS)
    calculate_window(df_aggregated_valid_target, t_valid, t_gt_valid, win_size, step_size, FROM, TO, START, END, AXIS)
    calculate_window(df_aggregated_test_target, t_test, t_gt_test, win_size, step_size, FROM, TO, START, END, AXIS)
    
elif task == 'Cross-Person':
    # Create an empty DataFrame to hold the aggregated data for this source position
    df_aggregated_train = pd.DataFrame()
    df_aggregated_valid = pd.DataFrame()
    df_aggregated_test = pd.DataFrame()
    for i in range(len(sources)):
        for position in position_list:
            for split_index in range(0, 3):
                file_name = person_array[sources[i]] + "_" + position + '_' + item[split_index]
                print(f"Processing: {file_name}")
                df = pd.read_csv(dataset_path + file_name + '.csv', sep=",")
                
                # Aggregate data into the corresponding split DataFrame
                if split_index == 0:
                    df_aggregated_train = pd.concat([df_aggregated_train, df], ignore_index=True)
                elif split_index == 1:
                    df_aggregated_valid = pd.concat([df_aggregated_valid, df], ignore_index=True)
                elif split_index == 2:
                    df_aggregated_test = pd.concat([df_aggregated_test, df], ignore_index=True)
        
    # Pass the aggregated data to calculate_window
    calculate_window(df_aggregated_train, s_train, s_gt_train, win_size, step_size, FROM, TO, START, END, AXIS)
    calculate_window(df_aggregated_valid, s_valid, s_gt_valid, win_size, step_size, FROM, TO, START, END, AXIS)
    calculate_window(df_aggregated_test, s_test, s_gt_test, win_size, step_size, FROM, TO, START, END, AXIS)

    # For the target position, similarly aggregate data across all users
    
    for i in range(len(target)):
        df_aggregated_train_target = pd.DataFrame()
        df_aggregated_valid_target = pd.DataFrame()
        df_aggregated_test_target = pd.DataFrame()
        
        for position in position_list:
            for split_index in range(0, 3):
                file_name = person_array[target[i]] + "_" + position + '_' + item[split_index]
                print(f"Processing: {file_name}")
                df = pd.read_csv(dataset_path + file_name + '.csv', sep=",")

                if split_index == 0:
                    df_aggregated_train_target = pd.concat([df_aggregated_train_target, df], ignore_index=True)
                elif split_index == 1:
                    df_aggregated_valid_target = pd.concat([df_aggregated_valid_target, df], ignore_index=True)
                elif split_index == 2:
                    df_aggregated_test_target = pd.concat([df_aggregated_test_target, df], ignore_index=True)

        calculate_window(df_aggregated_train_target, t_train, t_gt_train, win_size, step_size, FROM, TO, START, END, AXIS)
        calculate_window(df_aggregated_valid_target, t_valid, t_gt_valid, win_size, step_size, FROM, TO, START, END, AXIS)
        calculate_window(df_aggregated_test_target, t_test, t_gt_test, win_size, step_size, FROM, TO, START, END, AXIS)

elif task == "Cross-Device":
    pass

Source
Processing: User1_RA_train
Processing: User1_RA_valid
Processing: User1_RA_test
Processing: User2_RA_train
Processing: User2_RA_valid
Processing: User2_RA_test
Processing: User3_RA_train
Processing: User3_RA_valid
Processing: User3_RA_test
Processing: User4_RA_train
Processing: User4_RA_valid
Processing: User4_RA_test
Processing: User5_RA_train
Processing: User5_RA_valid
Processing: User5_RA_test
Processing: User6_RA_train
Processing: User6_RA_valid
Processing: User6_RA_test
Processing: User7_RA_train
Processing: User7_RA_valid
Processing: User7_RA_test
Processing: User8_RA_train
Processing: User8_RA_valid
Processing: User8_RA_test
Processing: User1_LA_train
Processing: User1_LA_valid
Processing: User1_LA_test
Processing: User2_LA_train
Processing: User2_LA_valid
Processing: User2_LA_test
Processing: User3_LA_train
Processing: User3_LA_valid
Processing: User3_LA_test
Processing: User4_LA_train
Processing: User4_LA_valid
Processing: User4_LA_test
Processing: User5_LA_train
Proces

In [48]:
# Number of rows or number of training domains (different positions)
print(type(s_train[0][0]))
print(len(s_train))
print(s_train[0][0].shape)

<class 'numpy.ndarray'>
89993
(128, 1, 3)


In [49]:
# Converting data list to numpy data

# Source
s_train = np.concatenate(s_train, axis=0).astype(np.float32)
s_gt_train = np.array(s_gt_train).astype(np.float32)
s_valid = np.concatenate(s_valid, axis=0).astype(np.float32)
s_gt_valid = np.array(s_gt_valid).astype(np.float32)
s_test = np.concatenate(s_test, axis=0).astype(np.float32)
s_gt_test = np.array(s_gt_test).astype(np.float32)

#======================================================================

# Target:
t_train = np.concatenate(t_train, axis=0).astype(np.float32)
t_gt_train = np.array(t_gt_train).astype(np.float32)
t_valid = np.concatenate(t_valid, axis=0).astype(np.float32)
t_gt_valid = np.array(t_gt_valid).astype(np.float32)
t_test = np.concatenate(t_test, axis=0).astype(np.float32)
t_gt_test = np.array(t_gt_test).astype(np.float32)

# Network

In [51]:
import torch.nn.init as init

def weight_init(m):
    '''
    Usage:
        model = Model()
        model.apply(weight_init)
    '''
    if isinstance(m, nn.Conv1d):
        init.normal_(m.weight.data)
        if m.bias is not None:
            init.normal_(m.bias.data)
    elif isinstance(m, nn.Conv2d):
        init.xavier_normal_(m.weight.data)
        if m.bias is not None:
            init.normal_(m.bias.data)
    elif isinstance(m, nn.Conv3d):
        init.xavier_normal_(m.weight.data)
        if m.bias is not None:
            init.normal_(m.bias.data)
    elif isinstance(m, nn.ConvTranspose1d):
        init.normal_(m.weight.data)
        if m.bias is not None:
            init.normal_(m.bias.data)
    elif isinstance(m, nn.ConvTranspose2d):
        init.xavier_normal_(m.weight.data)
        if m.bias is not None:
            init.normal_(m.bias.data)
    elif isinstance(m, nn.ConvTranspose3d):
        init.xavier_normal_(m.weight.data)
        if m.bias is not None:
            init.normal_(m.bias.data)
    elif isinstance(m, nn.BatchNorm1d):
        init.normal_(m.weight.data, mean=1, std=0.02)
        init.constant_(m.bias.data, 0)
    elif isinstance(m, nn.BatchNorm2d):
        init.normal_(m.weight.data, mean=1, std=0.02)
        init.constant_(m.bias.data, 0)
    elif isinstance(m, nn.BatchNorm3d):
        init.normal_(m.weight.data, mean=1, std=0.02)
        init.constant_(m.bias.data, 0)
    elif isinstance(m, nn.Linear):
        init.xavier_normal_(m.weight.data)
        init.normal_(m.bias.data)
    elif isinstance(m, nn.LSTM):
        for param in m.parameters():
            if len(param.shape) >= 2:
                init.orthogonal_(param.data)
            else:
                init.normal_(param.data)
    elif isinstance(m, nn.LSTMCell):
        for param in m.parameters():
            if len(param.shape) >= 2:
                init.orthogonal_(param.data)
            else:
                init.normal_(param.data)
    elif isinstance(m, nn.GRU):
        for param in m.parameters():
            if len(param.shape) >= 2:
                init.orthogonal_(param.data)
            else:
                init.normal_(param.data)
    elif isinstance(m, nn.GRUCell):
        for param in m.parameters():
            if len(param.shape) >= 2:
                init.orthogonal_(param.data)
            else:
                init.normal_(param.data)

In [52]:
from torch.nn.utils import spectral_norm
from torchgan.layers import MinibatchDiscrimination1d


class Classifier(nn.Module):
    """
        A no-frills convolutional classifier.
        SELU activation mitigates the need for batch normalization layers.
    """

    def __init__(self, n_channels=3, n_classes=6):
        """
            Initialize the classifier.
        :param n_channels:
        :param n_classes: number of activities to classify
        """
        super(Classifier, self).__init__()
        self.c = nn.Sequential(
            nn.Conv2d(in_channels=n_channels, out_channels=64, kernel_size=(9, 1)),
            nn.SELU(),
            nn.MaxPool2d(kernel_size=(2, 1)),

            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=(9, 1)),
            nn.SELU(),
            nn.MaxPool2d(kernel_size=(2, 1)),

            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=(9, 1)),
            nn.SELU(),
            nn.MaxPool2d(kernel_size=(2, 1)),

            nn.Flatten(),

            nn.Linear(in_features=256 * 9 * 1, out_features=64),
            nn.SELU(),

            nn.Linear(in_features=64, out_features=n_classes),
        )
        self.apply(weight_init)

    def forward(self, x):
        x = self.c(x)
        x = F.log_softmax(x, dim=1)
        return x


# x = torch.rand((32, 3, 128, 1))
# net_c = Classifier(n_classes=6, n_channels=3)
# net_c(x).shape



class Discriminator(nn.Module):
    """
        Spectral Normalization used for the discriminator Conv layers
        https://arxiv.org/pdf/1805.08318.pdf

        Mini-batch Discrimination used in FC layers
    """

    def __init__(self, n_channels=3):
        """
            Initialize the discriminator.
        :param n_channels: number of IMU channels, 3 if using only the accelerometer
        """
        super(Discriminator, self).__init__()
        self.d = nn.Sequential(
            spectral_norm(nn.Conv2d(in_channels=n_channels, out_channels=64, kernel_size=(9, 1))),
            nn.SELU(),
            nn.AvgPool2d(kernel_size=(2, 1)),

            spectral_norm(nn.Conv2d(in_channels=64, out_channels=128, kernel_size=(9, 1))),
            nn.SELU(),
            nn.AvgPool2d(kernel_size=(2, 1)),

            spectral_norm(nn.Conv2d(in_channels=128, out_channels=256, kernel_size=(9, 1))),
            nn.SELU(),
            nn.AvgPool2d(kernel_size=(2, 1)),

            nn.Flatten(),

            nn.Linear(in_features=256 * 9 * 1, out_features=64),
            nn.SELU(),

            MinibatchDiscrimination1d(64, 32),
            nn.Linear(in_features=96, out_features=1),
            # nn.Linear(in_features=64, out_features=1),
        )
        self.apply(weight_init)

    def forward(self, x):
        x = self.d(x)
        return x


# x = torch.rand((32, 3, 128, 1))
# net_d = Discriminator(n_channels=3)
# net_d(x).shape

class KMaxPooling(nn.Module):
    """
        K-max pooling layer that performs an argmax over the k largest entries
        along the last dimension. For more details
        https://discuss.pytorch.org/t/resolved-how-to-implement-k-max-pooling-for-cnn-text-classification/931/4
    """

    def __init__(self, k, dim):
        super(KMaxPooling, self).__init__()
        self.k = k
        self.dim = dim

    def forward(self, x):
        index = x.topk(self.k, dim=self.dim)[1].sort(dim=self.dim)[0]
        return x.gather(self.dim, index)


class SpatialTransformer(nn.Module):
    """
    The main building block for the Generator, the Spatial Transformer Block uses the localization network in tandem with
    the fully connected network to regress the 12 (4x3) parameters for the affine transformation that needs to be applied to
    the input sample
    """

    def __init__(self, n_channel=3, window_size=128):
        super(SpatialTransformer, self).__init__()

        self.n_channel = n_channel
        self.window_size = window_size

        self.n_conv1 = 64
        self.n_conv2 = 128
        self.n_fc1 = 64
        self.n_fc2 = 32
        self.n_pool = 8
        # self.n_pool = self.n_conv2 # when not doing any pooling,

        self.localization = nn.Sequential( # input:  (batch, 1, window, channel)
            # spectral_norm(
            nn.Conv2d(in_channels=1, out_channels=self.n_conv1, kernel_size=(1, 3)),  # output: (batch, 64, window, 1)
            nn.SELU(),
            # spectral_norm(
            nn.Conv2d(in_channels=self.n_conv1, out_channels=self.n_conv2, kernel_size=(1, 1)),  # output: (batch, 128, window, 1)
            nn.SELU(),
            KMaxPooling(self.n_pool, dim=2),  # output: (batch, 128, 4, 1)
            nn.Flatten()
            
        )

        self.fc_loc = nn.Sequential(  # input (batch, 128*8)
            nn.Linear(in_features=self.n_conv2 * self.n_pool, out_features=self.n_fc1),  # output: (batch, 64)
            nn.SELU(),
            nn.Linear(in_features=self.n_fc1, out_features=self.n_fc2),  # output: (batch, 32)
            nn.SELU(),
            nn.Linear(self.n_fc2, 12)
        )

        self.apply(weight_init)

        # ----------------------------------------------------------------------
        # need to make final fc layers weights zero and biases as identity
        # wrong initialization will lead to non convergence
        self.fc_loc[4].weight.data.zero_()
        self.fc_loc[4].bias.data.copy_(
            torch.tensor([1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0]))
        # ---------------------------------------------------------------------

    def forward(self, x):  # input:  (batch, channel, window, 1)
        x = x.permute(0, 3, 2, 1)  # output: (batch, 1, window, channel)

        xs = self.localization(x)  # output: (batch, 128*8, 1)
        theta = self.fc_loc(xs)  # output: (batch, 12)
        theta = theta.view(-1, 4, 3)  # output: (batch, 4, 3)

        x = x.squeeze(1)  # output: (batch, window, channel)
        ones = torch.ones(x.shape[0], x.shape[1], 1, dtype=torch.float, requires_grad=False).to(x.device)
        aug = torch.cat([x, ones], 2)  # output: (batch, window, channel+1)

        x = torch.matmul(aug, theta)  # output: (batch, window, channel)
        x = x.unsqueeze(1)  # output: (batch, 1, window, channel)

        return x.permute(0, 3, 2, 1).contiguous(), theta  # output: (batch, channel, window, 1)

# x = torch.rand((32, 1, 128, 3))
# LocalizationNetKMaxPool()(x).shape

In [ ]:
# Follow the tutorial
# ========================
# Hyper parameters
# ========================
seed               = 123
LR_G               = 0.0002
LR_D               = 0.0009
LR_C               = 0.001
MOMENTUM           = 0.9
beta1              = 0.9
beta2              = 0.99
latent_dim         = 32
N_EPOCH            = 20
N_EPOCH_CLF        = 20
BATCH_SIZE         = 32
GAMMA              = 0.95
soft_label_fake    = 0.1
soft_label_valid_disc = 0.9
soft_label_valid_gen  = 0.9

# num_workers        = 2

# dis_batch_size     = 64
# gen_batch_size     = 128
# max_epoch          = 250
# lambda_kld         = 1e-6
# cont_dim           = 16
# cont_k             = 8192
# cont_temp          = 0.07

S_train = [[] for _ in range(nsources)]
S_valid = [[] for _ in range(nsources)]
S_test = [[] for _ in range(nsources)]

T_train = []
T_valid = []
T_test = []

drop_last = True

source_train = InfiniteDataset(s_train)
target_train = InfiniteDataset(t_train)

source_loader_da = InfiniteDataLoader(source_train, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                                      num_workers=NUM_WORKERS, generator=rng, worker_init_fn=seed_worker)

target_loader_da = InfiniteDataLoader(target_train, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                                      num_workers=NUM_WORKERS, generator=rng, worker_init_fn=seed_worker)

S_train_clf, S_valid_clf, _ = load_train_valid_test(s_train, s_gt_train, s_valid, s_gt_valid, s_test, s_gt_test,
                                                    NUM_WORKERS, rng, seed_worker, batch_size=BATCH_SIZE)
_, T_valid_clf, T_test_clf = load_train_valid_test(t_train, t_gt_train, t_valid, t_gt_valid, t_test, t_gt_test,
                                                   NUM_WORKERS, rng, seed_worker, batch_size=BATCH_SIZE)

# Training the Spatial Transformer

In [53]:
log_filename = save_path + "training_log.txt"
# Configure logging
logfile = open(log_filename, 'w')
generator_save_path = save_path + "generator.pth"
discriminator_save_path = save_path + "discriminator.pth"

#===========================
# Declaring iterators
#===========================
T_train_itr = iter(target_loader_da)
T_valid_itr = iter(T_valid_clf)
S_train_itr = iter(source_loader_da)
S_valid_itr = iter(S_valid_clf)

# ==========================
# Initialize Network
# ==========================
generator = SpatialTransformer().to(DEVICE)
discriminator = Discriminator().to(DEVICE)

# =========================
# Initialize Optimizers
# =========================
optim_disc = SGD(discriminator.parameters(), lr=LR_D, weight_decay=1e-6, momentum=MOMENTUM)
optim_gen = Adam(generator.parameters(), lr=LR_G, betas=(LR_G_B1, LR_G_B2), amsgrad=False, weight_decay=1e-6)

#============================
# Loss
#============================
adversarial_loss = torch.nn.BCEWithLogitsLoss(reduction='mean').to(DEVICE)
recon_loss = nn.SmoothL1Loss().to(DEVICE)


#============================
# Training
#============================
# Taking the max for source and target loader da since they are from InfiniteDataloader
# T_valid_clf and s_valid_clf are not
n_batch = max(len(target_loader_da), len(source_loader_da))
n_valid_batch = min(len(T_valid_clf), len(S_valid_clf))

# Iterate over each epoch
for epoch in tqdm.tqdm(range(N_EPOCH), desc='total progress'):
    generator.train()
    discriminator.train()

    # Initialize accumulators for losses and accuracy
    gen_losstotal = 0.0
    disc_losstotal = 0.0
    correct_total = 0.0
    train_total = 0.0
    
    # Iterate over each batch
    for iter_idx in range(n_batch):
        # For source batch
        # Fetch a batch from the source training iterator
        # Since it it from InfininteDataLoader we do not pass any activity labels
        s_sample = next(S_train_itr)
#         except StopIteration:
#             # If iterator is exhausted, create a new iterator and fetch the next batch
#             S_train_itr = iter(source_loader_da)
#             s_sample, s_label = next(S_train_itr)
        # Move target samples to the specified device and cast to appropriate types
        s_sample = s_sample.to(DEVICE).float()
        
        # For target batch
        
        t_sample = next(T_train_itr)
#         except:
#             T_train_itr = iter(target_loader_da)
#             t_sampleAug, t_sample, _ = next(T_train_itr)
        t_sample = t_sample.to(DEVICE).float()
        
        #----------------------------
        # Train generator
        #----------------------------
        optim_gen.zero_grad()
        # Generate fake samples from target samples
        x_gen, _ = generator(t_sample)
        # Generator(source) is required for reconstruction loss
        x_gen_source, _ = generator(s_sample)

        # loss(input, target)
        # Discriminator's prediction on generated samples
        disc_pred = discriminator(x_gen)
        # Adversarial loss: how well the generator fools the discriminator
        loss_adv = adversarial_loss(disc_pred, valid_alt)
        # Reconstruction loss: how well the generator reconstructs the source samples
        loss_rec = recon_loss(x_gen_source, s_sample)
        # Total loss
        gen_loss = loss_adv + loss_rec * GAMMA
        
        # Backpropagation and optimization for generator
        torch.use_deterministic_algorithms(False)
        gen_loss.backward()
        torch.use_deterministic_algorithms(True)
        optim_gen.step()
        
        #--------------------------
        # Train discriminator
        #--------------------------
        optim_disc.zero_grad()
        # Discriminator's prediction on real source samples
        pred_real = discriminator(s_sample)
        # Discriminator's prediction on fake samples generated from target samples
        # We do not want to update the generator on fake source (fake source transformed from target)  predicitons, hence detach() used
        pred_fake = discriminator(x_gen.detach())
        # Update accuracy metrics
        correct_total += ((pred_real.round().eq(valid.round()) * 1).sum().item() + (pred_fake.round().eq(fake.round()) * 1).sum().item())
        train_total += BATCH_SIZE
        # Adversarial loss for real and fake samples
        loss_real = adversarial_loss(pred_real, valid)
        loss_fake = adversarial_loss(pred_fake, fake)
        # Total discriminator loss
        loss_d = (loss_real + loss_fake) / 2

        # Backpropagation and optimization for discriminator
        loss_d.backward()
        optim_disc.step()
        
        # Accumulate total losses
        gen_losstotal += gen_loss
        disc_losstotal += loss_d
    
    # Validation after an epoch of training is finished
    generator.eval()
    discriminator.eval()
    # Making sure gradients are not computed for validation
    with torch.no_grad():
        valid_correct = 0.0
        valid_total = 0.0

        for vindx in range(n_valid_batch):
            try:
                vs_sample, _ = next(S_valid_itr)
            except StopIteration:
                S_valid_itr = iter(S_valid_clf)
                vs_sample, _ = next(S_valid_itr)
            vs_sample = vs_sample.to(DEVICE).float()
            
            try:
                vt_sample, _ = next(T_valid_itr)
            except StopIteration:
                T_valid_itr = iter(T_valid_clf)
                vt_sample, _ = next(T_valid_itr)
            vt_sample = vt_sample.to(DEVICE).float()

            # generating fake source from target data (valid split)
            xv_gen, _ = generator(vt_sample)
            # discriminator's prediction on fake source (valid split)
            vpred_fake = discriminator(xv_gen)
            # discriminator's prediction on real source (valid split)
            vpred_real = discriminator(vs_sample)

            valid_correct += ((vpred_real.round().eq(valid.round()) * 1).sum().item() + (vpred_fake.round().eq(fake.round()) * 1).sum().item())
            valid_total += BATCH_SIZE

    gen_losstotal /= iter_idx
    disc_losstotal /= iter_idx
    # each batch has batch_size of real and batch_size of fake samples
    # total number of samples is twice the number of batch size
    correct_total /= (train_total * 2)
    valid_correct /= (valid_total * 2)
    # Saving the losses and accuracies
    logfile.write(f'EPOCH: {epoch+1}\n')
    logfile.write(f'GEN LOSS: {gen_losstotal}\n')
    logfile.write(f'DISC LOSS: {disc_losstotal}\n')
    logfile.write(f'DISC TRAIN ACC: {correct_total}\n')
    logfile.write(f"DISC VALID ACC: {valid_correct}\n")
    print(f"EPOCH: {epoch+1}/{N_EPOCH} ----> DISC TRAIN ACC: {correct_total} & DISC VALID ACC: {valid_correct}")
    torch.save(generator.state_dict(), generator_save_path)
    torch.save(discriminator.state_dict(), discriminator_save_path)
logfile.close()

total progress:   0%|                                    | 0/20 [00:41<?, ?it/s]


StopIteration: 

In [ ]:
# Open the log file
log_filename = save_path + "training_log.txt"
with open(log_filename, 'r') as file:
    # Read lines from the file
    lines = file.readlines()
file.close()
# Initialize variables to store loss values
gen_loss_value = None
disc_loss_value = None
disc_acc_value = None

gen_losslist = []
disc_losslist = []
disc_acclist = []

# Iterate over lines and extract loss values
for line in lines:
    if 'GEN LOSS' in line:
        gen_loss_value = float(line.split(': ')[1])
        gen_losslist.append(gen_loss_value)
        
    elif 'DISC LOSS' in line:
        disc_loss_value = float(line.split(': ')[1])
        disc_losslist.append(disc_loss_value)
    
    elif 'DISC ACC' in line:
        disc_acc_value = float(line.split(': ')[1])
        disc_acclist.append(disc_acc_value)

epochs = list(range(1, N_EPOCH+1)) # range(1, 51) --> assumes the test is for 50 epochs

plt.plot(epochs, gen_losslist, label='generator loss')
plt.plot(epochs, disc_losslist, label='discriminator loss')
plt.plot(epochs, disc_acclist, label='discriminator accuracy')

plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.savefig(save_path + 'train_loss_acc.png')
plt.show()

# Traning Classifier with Source domain data

In [ ]:
log_filename_clf = save_path + "training_log_clf.txt"
classifier_save_path = save_path + "classifier.pth"
logfile = open(log_filename_clf, 'w')

# Initializing classifier and the loss
classifier = Classifier(n_channels=3, n_classes=len(activities)).to(DEVICE)
clf_loss_criterion = nn.NLLLoss().to(DEVICE)

# Optimizer
optim_clf = Adam(classifier.parameters(), lr=LR_C, betas=(LR_C_B1, LR_C_B2))

# Iterator
S_valid_clf_itr = iter(S_valid_clf)

for epoch in tqdm.tqdm(range(N_EPOCH_CLF), desc='total progress'):
    classifier.train()
    loss_clf_total = 0.0
    task_correct = 0.0
    task_total = 0.0
    
    for iter_idx, (s_sample, s_label) in enumerate(tqdm.tqdm(S_train_clf)):
        s_sample, s_label = s_sample.to(DEVICE).float(), s_label.to(DEVICE).long()
        
        output = classifier(s_sample)
        loss_clf = clf_loss_criterion(output, s_label)
        loss_clf_total += loss_clf.item()
        
        optim_clf.zero_grad()
        loss_clf.backward()
        optim_clf.step()
        
        _, task_pred = torch.max(output.data, dim=1)
        task_correct += (task_pred == s_label).sum()
        task_total += output.size(0)
        
    classifier.eval()
    with torch.no_grad():
        valid_source_loss = 0.0
        valid_target_loss = 0.0
        valid_source_total = 0
        valid_target_total = 0
        valid_source_correct = 0
        valid_target_correct = 0
        
        for vidx, (vtsample, vtlabel) in enumerate(T_valid_clf):
            vtsample, vtlabel = vtsample.to(DEVICE).float(), vtlabel.to(DEVICE).long()
            
            try:
                vssample, vslabel = next(S_valid_clf_itr)
            except StopIteration:
                S_valid_clf_itr = iter(S_valid_clf)
                vssample, vslabel = next(S_valid_clf_itr)
            vssample, vslabel = vssample.to(DEVICE).float(), vslabel.to(DEVICE).long()
            
            
            outputvs = classifier(vssample)
            outputvt = classifier(vtsample)
            
            vsLoss = clf_loss_criterion(outputvs, vslabel)
            vtLoss = clf_loss_criterion(outputvt, vtlabel)
            
            valid_source_loss += vsLoss
            valid_target_loss += vtLoss
            valid_source_total += vssample.size(0)
            valid_target_total += vtsample.size(0)
            
            _, valid_source_pred = torch.max(outputvs.data, dim=1)
            valid_source_correct += (valid_source_pred == vslabel).sum()
            
            _, valid_target_pred = torch.max(outputvt.data, dim=1)
            valid_target_correct += (valid_target_pred == vtlabel).sum()
    
    train_acc = float(task_correct)*100/task_total
    valid_source_acc = float(valid_source_correct)*100/valid_source_total
    valid_target_acc = float(valid_target_correct)*100/valid_target_total
    
    print(f'EPOCH: {epoch+1}/{N_EPOCH_CLF}')
    print(f'TRAIN (SOURCE) LOSS: {loss_clf_total}')
    print(f'VALID SOURCE LOSS: {valid_source_loss}')
    print(f'VALID TARGET LOSS: {valid_target_loss}')
    print(f'TRAIN (SOURCE) ACC: {train_acc}')
    print(f'VALID SOURCE ACC: {valid_source_acc}')
    print(f'VALID TARGET ACC: {valid_target_acc}')
    
    logfile.write(f'EPOCH: {epoch+1}/{N_EPOCH_CLF}\n')
    logfile.write(f'TRAIN LOSS: {loss_clf_total}\n')
    logfile.write(f'VALID SOURCE LOSS: {valid_source_loss}\n')
    logfile.write(f'VALID TARGET LOSS: {valid_target_loss}\n')
    logfile.write(f'TRAIN ACC: {train_acc}\n')
    logfile.write(f'VALID SOURCE ACC: {valid_source_acc}\n')
    logfile.write(f'VALID TARGET ACC: {valid_target_acc}\n')
    torch.save(classifier.state_dict(), classifier_save_path)
logfile.close()

In [ ]:
#=======================================
# Loss and Accuracy of the Classifier
#=======================================

# Open the log file
log_filename = save_path + "training_log_clf.txt"
with open(log_filename, 'r') as file:
    # Read lines from the file
    lines = file.readlines()
file.close()
# Initialize variables to store loss values
train_loss_value = None
valid_loss_value = None
train_acc = None
valid_acc = None

clf_train_losslist = []
clf_valid_losslist = []
clf_train_acclist = []
clf_valid_acclist = []

# Iterate over lines and extract loss values
for line in lines:
    if 'TRAIN LOSS' in line:
        train_loss_value = float(line.split(': ')[1])
        clf_train_losslist.append(train_loss_value)
    elif 'VALID LOSS' in line:
        valid_loss_value = float(line.split(': ')[1])
        clf_valid_losslist.append(valid_loss_value)
    elif 'TRAIN ACC' in line:
        train_acc = float(line.split(': ')[1])
        clf_train_acclist.append(train_acc)
    elif 'VALID ACC' in line:
        valid_acc = float(line.split(': ')[1])
        clf_valid_acclist.append(valid_acc)

epochs = list(range(1, N_EPOCH_CLF+1)) # range(1, 51) --> assumes the test is for 50 epochs

# Loss Plot
plt.plot(epochs, clf_train_losslist, label='classifier train loss')
plt.plot(epochs, clf_valid_losslist, label='classifier valid loss')

plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

# Accuracy Plot
plt.plot(epochs, clf_train_acclist, label='classifier train accuracy')
plt.plot(epochs, clf_valid_acclist, label='classifier valid accuracy')

plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# Evaluation of the classifier on Transfromed Data

In [ ]:
test_file = open(save_path + 'result.txt', 'w')

classifier = Classifier(n_channels=3, n_classes=len(activities)).to(DEVICE)
classifier.load_state_dict(torch.load(classifier_save_path))
generator = SpatialTransformer().to(DEVICE)
generator.load_state_dict(torch.load(generator_save_path))

classifier.eval()
generator.eval()
task_correct = 0
task_total = 0
all_preds = []
all_labels = []

for vidx, (_, testSample, testLabel) in enumerate(T_test):
    testSample, testLabel = testSample.to(DEVICE).float(), testLabel.to(DEVICE).long()

    transformed, _ = generator(testSample)
    pred = classifier(transformed)

    _, task_pred = torch.max(pred.data, dim=1)
    all_preds.extend(task_pred.cpu().numpy())
    all_labels.extend(testLabel.cpu().numpy())

    task_correct += (task_pred == testLabel).sum()
    task_total += pred.size(0)

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
testAcc = float(task_correct) * 100 / task_total
precision = precision_score(all_labels, all_preds, average='micro')
recall = recall_score(all_labels, all_preds, average='micro')
f1 = f1_score(all_labels, all_preds, average='micro')


if task == "Cross-Position":
        print(f"Dataset: {dataset}, all persons")
        print(f"Test_accuracy_target-{domains[target]}: {testAcc}")
        print(f"Test_precision_target-{domains[target]}: {precision}")
        print(f"Test_recall_target-{domains[target]}: {recall}")
        print(f"Test_f1_target-{domains[target]}: {f1}")

        test_file.write(f"Dataset: {dataset}, all persons\n")
        test_file.write(f"Test_accuracy_target-{domains[target]}:{testAcc}\n")
        test_file.write(f"Test_precision_target-{domains[target]}:{precision}\n")
        test_file.write(f"Test_recall_target-{domains[target]}:{recall}\n")
        test_file.write(f"Test_f1_target-{domains[target]}:{f1}\n")
        
elif task == "Cross-Person":
        target_string = ", ".join([domains[i] for i in target])
        print(f"Dataset: {dataset}, all Positions")
        print(f"Test_accuracy_target-({target_string}): {testAcc}")
        print(f"Test_precision_target-({target_string}): {precision}")
        print(f"Test_recall_target-({target_string}): {recall}")
        print(f"Test_f1_target-({target_string}): {f1}")

        test_file.write(f"Dataset: {dataset}, all positions\n")
        test_file.write(f"Test_accuracy_target-({target_string}):{testAcc}\n")
        test_file.write(f"Test_precision_target-({target_string}):{precision}\n")
        test_file.write(f"Test_recall_target-({target_string}):{recall}\n")
        test_file.write(f"Test_f1_target-({target_string}):{f1}\n")

test_file.close()

# Visual Evaluation of the Transformed Data

In [ ]:
# # Reinitializing the DataLoaders because for training the entries (window data) were shuffled
# # We want subsequent window-data for comparison with the source data aka evaluation
# drop_last = True
# shuffle = False
# valid_batch_size = 32
# for i in range(nsources):
#     _, S_valid[i], S_test[i] = modified_load_train_valid_test(s_train[i], s_gt_train[i], s_valid[i], s_gt_valid[i], s_test[i], s_gt_test[i], batch_size, drop_last, shuffle)

# _, T_valid, T_test = modified_load_train_valid_test(t_train, t_gt_train, t_valid, t_gt_valid, t_test, t_gt_test, valid_batch_size, drop_last, shuffle)

In [ ]:
# # Each test and valid have 128 entries in 4 batches that means 32 entries per batch
# # Batch size is (32, 3, 128, 1), the last 1 is dummy
# # each batch of size (3, 128, 1), where 3 is the channels (x, y, z), 128 is the window size
# # This one entry accounts for a single activity


# S_valid_itr = iter(S_valid[0])
# T_valid_itr = iter(T_valid)

# generator = SpatialTransformer().to(DEVICE)
# generator.load_state_dict(torch.load(generator_save_path))

# """
# standing->[0], lying-back->[1], ascending->[2], walking-parking-lot->[3],
# treadmill-running->[4], stepper-exercise->[5],
# cross-trainer-exercise->[6], rowing->[7], jumping->[8],  playing-basketball->[9]
# """

# activity_to_test = 2

# # data entries based on activity
# t_test_activity = None
# t_valid_activity = None

# s_test_activity = None
# s_valid_activity = None

# # Extracting transformed data based on activity labels
# # t_tsample --> target test, t_vsample --> target valid

# #---------------------------
# # For the Target
# #---------------------------
# for itr_idx, (t_tsampleAug, t_tsample, t_tlabel) in enumerate(tqdm.tqdm(T_test)):
#     _, t_tsample, t_tlabel = t_tsampleAug.to(DEVICE).float(), t_tsample.to(DEVICE).float(), t_tlabel.to(DEVICE).long()
#     try:
#         t_vsampleAug, t_vsample, t_vlabel = next(T_valid_itr)
#     except StopIteration:
#         T_valid_itr = iter(T_valid)
#         t_vsampleAug, t_vsample, t_vlabel = next(T_valid_itr)
#     _, t_vsample, t_vlabel = t_vsampleAug.to(DEVICE).float(), t_vsample.to(DEVICE).float(), t_vlabel.to(DEVICE).long()
    
#     test_trans, _ = generator(t_tsample)
#     valid_trans, _ = generator(t_vsample)
    
#     # for test data
#     indices = torch.nonzero(t_tlabel == activity_to_test).squeeze()
#     if t_test_activity is None:
#         t_test_activity = torch.index_select(test_trans, dim=0, index=indices)
#     else:
#         t_test_activity = torch.cat((t_test_activity, torch.index_select(test_trans, dim=0, index=indices)), dim=0)
    
#     # for validation data
#     indices = torch.nonzero(t_vlabel == activity_to_test).squeeze()
#     if t_valid_activity is None:
#         t_valid_activity = torch.index_select(valid_trans, dim=0, index=indices)
#     else:
#         t_valid_activity = torch.cat((t_valid_activity, torch.index_select(valid_trans, dim=0, index=indices)), dim=0)

        
# #-----------------------------------------------------------------------
# # P.S: t_test_activity and t_valid_activity hold the transformed data.
# #-----------------------------------------------------------------------
# # For the Source 
# #----------------------------
# for itr_idx, (s_tsampleAug, s_tsample, s_tlabel) in enumerate(tqdm.tqdm(S_test[0])):
#     _, s_tsample, s_tlabel = s_tsampleAug.to(DEVICE).float(), s_tsample.to(DEVICE).float(), s_tlabel.to(DEVICE).long()
#     try:
#         s_vsampleAug, s_vsample, s_vlabel = next(S_valid_itr)
#     except StopIteration:
#         S_valid_itr = iter(S_valid[0])
#         s_vsampleAug, s_vsample, s_vlabel = next(S_valid_itr)
#     _, s_vsample, s_vlabel = s_vsampleAug.to(DEVICE).float(), s_vsample.to(DEVICE).float(), s_vlabel.to(DEVICE).long()
    
#     # for test data
#     indices = torch.nonzero(s_tlabel == activity_to_test).squeeze()
#     if s_test_activity is None:
#         s_test_activity = torch.index_select(s_tsample, dim=0, index=indices)
#     else:
#         s_test_activity = torch.cat((s_test_activity, torch.index_select(s_tsample, dim=0, index=indices)), dim=0)
    
#     # for validation data
#     indices = torch.nonzero(s_vlabel == activity_to_test).squeeze()
#     if s_valid_activity is None:
#         s_valid_activity = torch.index_select(s_vsample, dim=0, index=indices)
#     else:
#         s_valid_activity = torch.cat((s_valid_activity, torch.index_select(s_vsample, dim=0, index=indices)), dim=0)

        
# # Getting rid of the dummy last axis        
# t_test_activity = t_test_activity.squeeze(3)
# t_valid_activity = t_valid_activity.squeeze(3)
# s_test_activity = s_test_activity.squeeze(3)
# s_valid_activity = s_valid_activity.squeeze(3)
# #output size: (freq_of_the_activity, 3, 128), where 3-->channel, 128-->window

# # t_valid_activity and s_valid_activity may not have equal number of entries
# #-------------------------------------------
# # So after running the above we have the
# # t_test_activity and t_valid_activity
# # s_test_activity and s_valid_activity
# #-------------------------------------------


# # x/y_target_test/valid, they do not represent the target domain itself, but rather, the transformed domain
# # target domain --> source domain
# x_target_test = []
# x_source_test = []
# x_target_valid = []
# x_source_valid = []

# y_target_test = []
# y_source_test = []
# y_target_valid = []
# y_source_valid = []

# z_target_test = []
# z_source_test = []
# z_target_valid = []
# z_source_valid = []

# # extracting x axis values
# x_target_test = t_test_activity[:, 0, :]
# x_target_valid = t_valid_activity[:, 0, :]
# x_source_test = s_test_activity[:, 0, :]
# x_source_valid = s_valid_activity[:, 0, :]

# y_target_test = t_test_activity[:, 1, :]
# y_target_valid = t_valid_activity[:, 1, :]
# y_source_test = s_test_activity[:, 1, :]
# y_source_valid = s_valid_activity[:, 1, :]

# z_target_test = t_test_activity[:, 2, :]
# z_target_valid = t_valid_activity[:, 2, :]
# z_source_test = s_test_activity[:, 2, :]
# z_source_valid = s_valid_activity[:, 2, :]

# # print(x_target_test)

In [ ]:
# for itr_idx, (t_tsampleAug, t_tsample, t_tlabel) in enumerate(tqdm.tqdm(T_test)):
#     print(t_tlabel)
#     break

## X-Axis

In [ ]:
# x_target_test_list = x_target_test.flatten().tolist()
# x_target_valid_list = x_target_valid.flatten().tolist()
# x_source_test_list = x_source_test.flatten().tolist()
# x_source_valid_list = x_source_valid.flatten().tolist()

# x_target_test_time = list(range(1, len(x_target_test_list)+1))
# x_source_test_time = list(range(1, len(x_source_test_list)+1))

# plt.plot(x_target_test_time, x_target_test_list, label='Transformed source domain')
# plt.plot(x_source_test_time, x_source_test_list, label='Original source domain')

# plt.title(f'transformed VS source domain, Activity: ({activities[activity_to_test]})')
# plt.xlabel('Timesteps')
# plt.ylabel('x-axis Accelerometer')
# plt.legend()
# plt.show()

## Y-Axis

In [ ]:
# y_target_test_list = y_target_test.flatten().tolist()
# y_target_valid_list = y_target_valid.flatten().tolist()
# y_source_test_list = y_source_test.flatten().tolist()
# y_source_valid_list = y_source_valid.flatten().tolist()

# y_target_test_time = list(range(1, len(y_target_test_list)+1))
# y_source_test_time = list(range(1, len(y_source_test_list)+1))

# plt.plot(x_target_test_time, y_target_test_list, label='Transformed source domain')
# plt.plot(x_source_test_time, y_source_test_list, label='Original source domain')

# plt.title(f'transformed VS source domain, Activity: ({activities[activity_to_test]})')
# plt.xlabel('Timesteps')
# plt.ylabel('y-axis Accelerometer')
# plt.legend()
# plt.show()

## Z-Axis

In [ ]:
# z_target_test_list = z_target_test.flatten().tolist()
# z_target_valid_list = z_target_valid.flatten().tolist()
# z_source_test_list = z_source_test.flatten().tolist()
# z_source_valid_list = z_source_valid.flatten().tolist()

# z_target_test_time = list(range(1, len(z_target_test_list)+1))
# z_source_test_time = list(range(1, len(z_source_test_list)+1))

# plt.plot(z_target_test_time, z_target_test_list, label='Transformed source domain')
# plt.plot(z_source_test_time, z_source_test_list, label='Original source domain')

# plt.title(f'transformed VS source domain, Activity: ({activities[activity_to_test]})')
# plt.xlabel('Timesteps')
# plt.ylabel('z-axis Accelerometer')
# plt.legend()
# plt.show()

# =========================  END  =============================
## The below part is not a part of this project

# Scrapping

In [ ]:
# labelcounter = 0
# for itr_idx, (t_tsampleAug, t_tsample, t_tlabel) in enumerate(tqdm.tqdm(T_valid)):
#     labelcounter += t_tlabel.size(0)
# print(labelcounter)

In [ ]:
# log_filename = base_url + "/Saves/dsads_Contras_ContrasAug_Recon_ReconAug_Log.txt"
# # Read loss values from the log file
# with open(log_filename, 'r') as file:
#     lines = file.readlines()

# # Extract loss values from the lines
# loss_values = [float(line.split()[-1]) for line in lines]

# # Plotting the loss values
# plt.figure(figsize=(10, 6))
# plt.plot(range(1, len(loss_values) + 1), loss_values, marker='o', linestyle='-')
# plt.title('Training Loss Over Epochs')
# plt.xlabel('Epoch')
# plt.ylabel('Loss')
# plt.grid(True)
# plt.show()
# file.close()

In [ ]:
# for iter_idx, (sampleAug, sample, label) in enumerate(tqdm.tqdm(S_train[0])):
#     print(sample.shape)
#     print(sampleAug.shape)
#     break

In [ ]:
# valid = torch.ones(batch_size, 1, requires_grad=False).to(DEVICE) * 0.9
# print(valid.round())

In [ ]:
# #======================================
# # label for real data in discriminator
# #======================================
# valid = torch.ones(batch_size, 1, requires_grad=False).to(DEVICE) * soft_label_valid_disc
# fake = torch.ones(batch_size, 1, requires_grad=False).to(DEVICE) * soft_label_fake
# valid_alt = torch.ones(batch_size, 1, requires_grad=False).to(DEVICE) * soft_label_valid_gen

# #======================================
# # Log File & Save Paths
# #======================================
# log_filename = save_path + 'training_log.txt'
# # Configure logging
# logfile = open(log_filename, 'w')
# generator_save_path = save_path + "generator.pth"
# discriminator_save_path = save_path + "discriminator.pth"

# # =========================
# # Initialize Network
# # =========================
# generator = SpatialTransformer().to(DEVICE)
# discriminator = Discriminator().to(DEVICE)

# # =========================
# # Initialize Optimizers
# # =========================
# optim_disc = SGD(discriminator.parameters(), lr=LR_D, weight_decay=1e-6, momentum=MOMENTUM)
# # 0.000002, momentum=0.9
# # self.optim_d = Adam(self.discriminator.parameters(),
# # lr=args.lr_FD, weight_decay=1e-6)  # 0.000002, momentum=0.9
# optim_gen = Adam(generator.parameters(), lr=LR_G, betas=(beta1, beta2), amsgrad=False, weight_decay=1e-6)
# # 0.0002

# #============================
# # Loss
# #============================
# adversarial_loss = torch.nn.BCEWithLogitsLoss(reduction='mean').to(DEVICE)
# recon_loss = nn.SmoothL1Loss().to(DEVICE)

# #============================
# # Initializing Iterator
# #============================
# # T_train_itr = iter(T_train)
# # T_valid_itr = iter(T_valid)
# # T_test_itr = iter(T_test)
# # S_train_itr = iter(S_train[0])

# #============================
# # Training
# #============================
# n_batch = len(S_train)
# generator.train()
# discriminator.train()

# for epoch in tqdm.tqdm(range(N_EPOCH), desc='total progress'):
#     gen_losstotal = 0.0
#     disc_losstotal = 0.0
#     correct_total = 0.0
#     train_total = 0.0
    
#     T_train_itr = iter(T_train)
#     T_valid_itr = iter(T_valid)
#     T_test_itr = iter(T_test)
#     S_train_itr = iter(source_loader_da)
    
#     for iter_idx in range(n_batch):
#         try:
#             s_sampleAug, s_sample, s_label = next(S_train_itr)
#         s_sampleAug, s_sample, s_label = s_sampleAug.to(DEVICE).float(), s_sample.to(DEVICE).float(), s_label.to(DEVICE).long()
        
# #         try:
#         t_sampleAug, t_sample, t_label = next(T_train_itr)
# #         except StopIteration:
# #             T_train_itr = iter(T_train)
# #             t_sampleAug, t_sample, t_label = next(T_train_itr)
#         t_sampleAug, t_sample, t_label = t_sampleAug.to(DEVICE).float(), t_sample.to(DEVICE).float(), t_label.to(DEVICE).long()
        
#         #----------------------------
#         # Train generator
#         #----------------------------
#         x_gen, _ = generator(t_sample)
#         x_gen_source, _ = generator(s_sample)
#         # loss(input, target)
#         disc_pred = discriminator(x_gen)
#         loss_adv = adversarial_loss(disc_pred, valid_alt)
#         loss_rec = recon_loss(x_gen_source, s_sample)
#         gen_loss = loss_adv + loss_rec * gamma
        
#         optim_gen.zero_grad()
#         gen_loss.backward()
#         optim_gen.step()
        
#         #--------------------------
#         # Train discriminator
#         #--------------------------
#         optim_disc.zero_grad()
#         pred_real = discriminator(s_sample)
#         pred_fake = discriminator(x_gen.detach())
#         correct_total += ((pred_real.round().eq(valid.round()) * 1).sum().item() + (pred_fake.round().eq(fake.round()) * 1).sum().item())
#         train_total += batch_size
#         loss_real = adversarial_loss(pred_real, valid)
#         loss_fake = adversarial_loss(pred_fake, fake)
#         loss_d = (loss_real + loss_fake) / 2
#         loss_d.backward()
#         optim_disc.step()
        
#         gen_losstotal += gen_loss
#         disc_losstotal += loss_d

#     gen_losstotal /= iter_idx
#     disc_losstotal /= iter_idx
#     correct_total /= (train_total * 2)
#     logfile.write(f'GEN LOSS: {gen_losstotal}\n')
#     logfile.write(f'DISC LOSS: {disc_losstotal}\n')
#     logfile.write(f'DISC ACC: {correct_total}\n')
#     torch.save(generator.state_dict(), generator_save_path)
#     torch.save(discriminator.state_dict(), discriminator_save_path)
# logfile.close()